# Member 3 — Model Fine-tuning (AISA-ArabicFC, Islamic Services Subset)

Objective: fine-tune a LoRA adapter that reproduces, then exceeds, the
official baseline `TuwaiqAcademy/AISA-AR-FunctionCall-Think` on the
3-tool Islamic subset (`get_qibla_direction`, `calculate_zakat`,
`search_quran`).

Baseline reference metrics (full dataset, per the dataset card):

| Metric | Baseline (AISA-Think) |
|---|---|
| FnAcc | 0.982 |
| ArgEM | 0.541 |
| ThinkRate | 0.868 |
| Overall (Track B) | 0.739 |

On the narrower 3-tool subset, these values represent a beatable floor
rather than a ceiling. ArgEM (argument extraction accuracy) is the metric
with the most documented headroom.

### Architectural basis

This notebook follows the AISA reference architecture for agentic AI
systems (Nacar et al., 2026, zenodo.org/records/18161880), which
separates a function-calling system into distinct layers:

- Foundational model layer — the base LLM being adapted.
- Tool layer — schema validation of generated calls (Step 4).
- Orchestration/training layer — prompt structuring, loss masking, SFT
  (Steps 5–7).
- Evaluation layer — structured, repeatable scoring (Step 8; Member 4's
  official scorer downstream).
- Deployment/governance layer — versioned checkpoints, logged runs,
  documented handoff (Step 9).

Each section below is labeled with the AISA layer it corresponds to.

## Step 0 — Install Dependencies

AISA layer: environment setup (prerequisite for all subsequent steps).

Package rationale:
- `transformers` — base model and tokenizer loading.
- `datasets` — JSONL/Arrow data handling.
- `peft` — LoRA implementation.
- `trl` — `SFTConfig`/`SFTTrainer`, providing completion-only loss
  masking via `completion_only_loss=True` on a prompt/completion dataset
  (see Step 6).
- `accelerate` — device placement and mixed precision support.
- `bitsandbytes` — required only for optional 4-bit/QLoRA training.

Note: the install cell also removes `torchao`, since `peft` raises an
`ImportError` if any version below its required threshold is present,
even though torchao-based quantization is unused here. A runtime restart
is required after this cell, before continuing to Step 1.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!unzip -q "TuwaiqAcademy-AISA-ArabicFC.zip" -d /content/

In [2]:
# Run once. Safe to re-run (pip is idempotent).
%pip install -q -U transformers datasets peft trl accelerate
%pip install -q -U bitsandbytes  # optional: only used if you enable 4-bit (QLoRA)
# peft raises an ImportError if *any* torchao version below 0.16 is present,
# even though we never use torchao-based quantization. Some environments
# (e.g. Colab) ship an older torchao that can't always be upgraded past that
# threshold due to the pinned torch version. Simplest fix: remove it
# entirely -- peft silently skips the check when torchao isn't installed.
%pip uninstall -y -q torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00


## Step 1 — Configure Data Paths

AISA layer: data / foundational inputs.

Loads existing outputs from Members 1 and 2: `aisa_islamic_train/`,
`aisa_islamic_dev/` (Arrow format, via `save_to_disk`), `Database/*.parquet`
(full raw splits prior to Islamic-subset filtering), and
`formatted/*.jsonl` (SFT-ready files). No re-downloading or reprocessing
required.

In [3]:
from pathlib import Path

# Hardcoded to your confirmed Colab path (data lives inside the unzipped
# repo folder). If you ever unzip somewhere else, or a teammate's folder
# name differs, run:  !find /content -maxdepth 4 -iname "formatted"
# and update this line to match.
PROJECT_ROOT = Path("/content/TuwaiqAcademy-AISA-ArabicFC")

PATHS = {
    "islamic_train_arrow": PROJECT_ROOT / "aisa_islamic_train",
    "islamic_dev_arrow":   PROJECT_ROOT / "aisa_islamic_dev",
    "formatted_dir":       PROJECT_ROOT / "formatted",
    "track_a_train":       PROJECT_ROOT / "formatted" / "islamic_train_track_a.jsonl",
    "track_b_train":       PROJECT_ROOT / "formatted" / "islamic_train_track_b.jsonl",
    "dev_eval":            PROJECT_ROOT / "formatted" / "islamic_dev_eval.jsonl",
    "stats":               PROJECT_ROOT / "formatted" / "stats.json",
}

for name, p in PATHS.items():
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {name}: {p}")


[MISSING] islamic_train_arrow: /content/TuwaiqAcademy-AISA-ArabicFC/aisa_islamic_train
[MISSING] islamic_dev_arrow: /content/TuwaiqAcademy-AISA-ArabicFC/aisa_islamic_dev
[MISSING] formatted_dir: /content/TuwaiqAcademy-AISA-ArabicFC/formatted
[MISSING] track_a_train: /content/TuwaiqAcademy-AISA-ArabicFC/formatted/islamic_train_track_a.jsonl
[MISSING] track_b_train: /content/TuwaiqAcademy-AISA-ArabicFC/formatted/islamic_train_track_b.jsonl
[MISSING] dev_eval: /content/TuwaiqAcademy-AISA-ArabicFC/formatted/islamic_dev_eval.jsonl
[MISSING] stats: /content/TuwaiqAcademy-AISA-ArabicFC/formatted/stats.json


## Step 2 — Data Validation

AISA layer: tool layer / data validation.

Verifies two conditions prior to training:
1. Only the 3 expected tools appear as gold `tool_called` values.
2. `calculate_inheritance` appears within `tools_sampled` candidate lists
   despite never being gold-labeled, confirming the clarification above
   against the actual dataset.

In [8]:
import json

ISLAMIC_GOLD_TOOLS = {"get_qibla_direction", "calculate_zakat", "search_quran"}

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_b = load_jsonl(PATHS["track_b_train"])
dev_eval = load_jsonl(PATHS["dev_eval"])

gold_tools_seen = {r["tool_called"] for r in train_b} | {r["tool_called"] for r in dev_eval}
print("Gold tool_called values seen:", gold_tools_seen)
assert gold_tools_seen <= (ISLAMIC_GOLD_TOOLS | {"none"}), (
    "Unexpected gold tool found — investigate before training!"
)

distractor_hits = sum(
    1 for r in (train_b + dev_eval)
    if "calculate_inheritance" in r.get("tools_sampled", [])
)
print(f"'calculate_inheritance' appears as a candidate distractor in "
      f"{distractor_hits} examples (expected > 0 per the clarification email).")

print(f"\nTrain (Track B): {len(train_b)} examples")
print(f"Dev eval:         {len(dev_eval)} examples")


Gold tool_called values seen: {'get_qibla_direction', 'calculate_zakat', 'search_quran'}
'calculate_inheritance' appears as a candidate distractor in 187 examples (expected > 0 per the clarification email).

Train (Track B): 1525 examples
Dev eval:         73 examples


## Step 3 — Loss-Monitoring Dev Split

AISA layer: orchestration / training-data contract.

`islamic_dev_eval.jsonl` provides gold labels (`tool_called`, `arguments`,
`think`) without a gold `text` completion field, as it is intended for
Member 4's inference-based scoring rather than `eval_loss` computation. A
held-out slice is therefore carved from `islamic_train_track_b.jsonl`
(which includes full `text`), with training conducted on the remainder.
This keeps loss evaluation consistent with the actual training data.

In [9]:
import random

random.seed(42)
shuffled = train_b[:]
random.shuffle(shuffled)

n_holdout = max(20, int(0.1 * len(shuffled)))  # ~10%, at least 20 examples
loss_eval_split = shuffled[:n_holdout]
train_split = shuffled[n_holdout:]

print(f"Training on {len(train_split)} examples, "
      f"holding out {len(loss_eval_split)} for eval_loss monitoring.")

FORMATTED_DIR = PATHS["formatted_dir"]
train_split_path = FORMATTED_DIR / "islamic_train_track_b_trainsplit.jsonl"
loss_eval_path = FORMATTED_DIR / "islamic_train_track_b_lossevalsplit.jsonl"

def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(train_split_path, train_split)
write_jsonl(loss_eval_path, loss_eval_split)
print("Wrote:", train_split_path)
print("Wrote:", loss_eval_path)


Training on 1373 examples, holding out 152 for eval_loss monitoring.
Wrote: /content/TuwaiqAcademy-AISA-ArabicFC/formatted/islamic_train_track_b_trainsplit.jsonl
Wrote: /content/TuwaiqAcademy-AISA-ArabicFC/formatted/islamic_train_track_b_lossevalsplit.jsonl


## Step 4 — Tool-Layer Output Validator

AISA layer: tool layer (schema validation of generated output).

Implements a lightweight validator to check generated output against the
tool registry rather than trusting raw model text directly, consistent
with AISA's tool-layer contract. Validation checks:
- Output parses against the `call:TOOL_NAME{...}` grammar.
- `TOOL_NAME` is one of the 3 valid Islamic tools.
- Whether `calculate_inheritance` was incorrectly called (hallucination
  check).

This provides a local, low-cost sanity check independent of Member 4's
official scoring pipeline.

In [10]:
import re

CALL_RE = re.compile(
    r"call:(?P<name>[a-zA-Z_]+)\{(?P<args>.*?)\}",
    re.DOTALL,
)

def parse_function_call(generated_text: str) -> dict:
    # Best-effort parse of a FunctionGemma-style call: call:TOOL_NAME{args}
    match = CALL_RE.search(generated_text)
    if not match:
        return {"parsed": False, "tool_name": None, "raw": generated_text}
    return {
        "parsed": True,
        "tool_name": match.group("name"),
        "args_raw": match.group("args"),
        "raw": generated_text,
    }

def validate_call(parsed: dict) -> dict:
    # Tool-layer validation: is this call legal for our Islamic subset?
    if not parsed["parsed"]:
        return {"valid_format": False, "valid_tool": False, "hallucinated_inheritance": False}
    tool_name = parsed["tool_name"]
    return {
        "valid_format": True,
        "valid_tool": tool_name in ISLAMIC_GOLD_TOOLS,
        "hallucinated_inheritance": tool_name == "calculate_inheritance",
    }

# Quick self-test
_test = parse_function_call("<think>...</think><start_function_call>call:calculate_zakat{amount:<escape>5000<escape>}<end_function_call>")
print(_test)
print(validate_call(_test))


{'parsed': True, 'tool_name': 'calculate_zakat', 'args_raw': 'amount:<escape>5000<escape>', 'raw': '<think>...</think><start_function_call>call:calculate_zakat{amount:<escape>5000<escape>}<end_function_call>'}
{'valid_format': True, 'valid_tool': True, 'hallucinated_inheritance': False}


## Step 5 — Baseline Snapshot

AISA layer: evaluation layer.

The unmodified baseline model is run on a sample of dev prompts prior to
training, establishing a documented "before" reference for later
comparison.

Note: loading `TuwaiqAcademy/AISA-AR-FunctionCall-Think` requires network
access to huggingface.co. If unavailable in the current environment, this
step may be skipped in favor of Member 4's official baseline numbers.

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "TuwaiqAcademy/AISA-AR-FunctionCall-Think"

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if device != "cpu" else torch.float32,
).to(device)
baseline_model.eval()
print("Baseline model loaded.")


Using device: cuda


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

Baseline model loaded.


In [12]:
@torch.no_grad()
def generate_call(model, tokenizer, prompt: str, max_new_tokens: int = 200) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    # Only the newly generated portion (after the prompt) matters.
    return full_text[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)):]

sample_prompts = [r["prompt"] for r in dev_eval[:10]]
sample_gold = [(r["tool_called"], r["arguments"]) for r in dev_eval[:10]]

baseline_results = []
for prompt, (gold_tool, gold_args) in zip(sample_prompts, sample_gold):
    gen = generate_call(baseline_model, tokenizer, prompt)
    parsed = parse_function_call(gen)
    valid = validate_call(parsed)
    baseline_results.append({
        "gold_tool": gold_tool,
        "predicted_tool": parsed["tool_name"],
        "correct_tool": parsed["tool_name"] == gold_tool,
        **valid,
    })

import pandas as pd
baseline_df = pd.DataFrame(baseline_results)
print(baseline_df)
print("\nBaseline quick FnAcc (10-sample, informal):",
      baseline_df["correct_tool"].mean())


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

             gold_tool       predicted_tool  correct_tool  valid_format  \
0  get_qibla_direction  get_qibla_direction          True          True   
1  get_qibla_direction  get_qibla_direction          True          True   
2      calculate_zakat      calculate_zakat          True          True   
3         search_quran         search_quran          True          True   
4      calculate_zakat      calculate_zakat          True          True   
5  get_qibla_direction  get_qibla_direction          True          True   
6      calculate_zakat      calculate_zakat          True          True   
7  get_qibla_direction  get_qibla_direction          True          True   
8  get_qibla_direction  get_qibla_direction          True          True   
9  get_qibla_direction  get_qibla_direction          True          True   

   valid_tool  hallucinated_inheritance  
0        True                     False  
1        True                     False  
2        True                     False  
3     

## Step 6 — LoRA Fine-Tuning Configuration

AISA layer: orchestration / training layer.

LoRA is used instead of full fine-tuning: it trains a small set of
low-rank adapter matrices (typically <5% of total parameters) injected
into attention/MLP projections, sufficient to specialize an already
function-call-capable model to this narrower 3-tool domain, with lower
hardware requirements than full fine-tuning.

`completion_only_loss=True` on a prompt/completion-structured dataset
restricts gradient updates to the `<think>...</think>` and function-call
tokens, excluding the prompt and tool declarations — consistent with the
completion-only masking used in the original AISA-AR-FunctionCall
training recipe. This is handled natively by `trl` given separate
`prompt` and `completion` columns, as provided by Member 2's JSONL.

In [13]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
# Note: DataCollatorForCompletionOnlyLM was removed from newer trl releases.
# Since our JSONL already has separate `prompt` and `completion` columns,
# SFTTrainer can mask the loss to the completion automatically (see
# `completion_only_loss=True` in SFTConfig below) -- no manual collator needed.

def jsonl_to_prompt_completion_dataset(path: Path) -> Dataset:
    records = load_jsonl(path)
    return Dataset.from_dict({
        "prompt": [r["prompt"] for r in records],
        "completion": [r["completion"] for r in records],
    })

train_ds = jsonl_to_prompt_completion_dataset(train_split_path)
loss_eval_ds = jsonl_to_prompt_completion_dataset(loss_eval_path)
print(train_ds)
print(loss_eval_ds)


Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1373
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 152
})


In [14]:
def build_lora_model(base_model_name: str, lora_r: int, lora_alpha: int, lora_dropout: float, use_4bit: bool = False):
    model_kwargs = {"torch_dtype": torch.bfloat16 if device != "cpu" else torch.float32}
    if use_4bit and device == "cuda":
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
    base = AutoModelForCausalLM.from_pretrained(base_model_name, **model_kwargs)
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    peft_model = get_peft_model(base, lora_config)
    peft_model.print_trainable_parameters()
    return peft_model


def train_one_run(run_name: str, lora_r: int, lora_alpha: int, lr: float,
                   epochs: float, batch_size: int, grad_accum: int,
                   base_model_name: str = BASE_MODEL, use_4bit: bool = False):
    output_dir = Path("runs") / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    model = build_lora_model(base_model_name, lora_r, lora_alpha, 0.05, use_4bit)

    sft_config = SFTConfig(
        output_dir=str(output_dir),
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        bf16=(device == "cuda"),
        gradient_checkpointing=(device == "cuda"),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_length=2048,
        completion_only_loss=True,
        report_to=[],
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=loss_eval_ds,
        processing_class=tokenizer,
    )

    trainer.train()
    eval_metrics = trainer.evaluate()

    best_dir = output_dir / "best_checkpoint"
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))

    return {
        "run_name": run_name,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lr": lr,
        "epochs": epochs,
        "eval_loss": eval_metrics.get("eval_loss"),
        "checkpoint_dir": str(best_dir),
    }


## Step 7 — Hyperparameter Sweep

AISA layer: orchestration / deployment (versioned, comparable runs).

A small grid of `(rank, learning rate, epochs)` combinations is trained
and compared by `eval_loss`, providing an evidence-based selection rather
than a single untested configuration. Checkpoints and results are saved
to persistent storage after each individual run to guard against session
interruption.

In [15]:
sweep_grid = [
    {"lora_r": 8,  "lora_alpha": 16, "lr": 2e-4, "epochs": 3},
    {"lora_r": 16, "lora_alpha": 32, "lr": 2e-4, "epochs": 3},
    {"lora_r": 16, "lora_alpha": 32, "lr": 1e-4, "epochs": 5},
]

# CPU-friendly small batch + more accumulation to keep effective batch size reasonable.
BATCH_SIZE = 2 if device == "cpu" else 4
GRAD_ACCUM = 16 if device == "cpu" else 8

sweep_results = []
for i, cfg in enumerate(sweep_grid):
    run_name = f"islamic_track_b_run{i}_r{cfg['lora_r']}_lr{cfg['lr']}"
    print(f"\n=== Starting run: {run_name} ===")
    result = train_one_run(
        run_name=run_name,
        lora_r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lr=cfg["lr"],
        epochs=cfg["epochs"],
        batch_size=BATCH_SIZE,
        grad_accum=GRAD_ACCUM,
    )
    sweep_results.append(result)
    print(result)

import pandas as pd
sweep_df = pd.DataFrame(sweep_results).sort_values("eval_loss")
sweep_df.to_csv("runs/sweep_results.csv", index=False)
sweep_df



=== Starting run: islamic_track_b_run0_r8_lr0.0002 ===


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

trainable params: 1,898,496 || all params: 269,996,672 || trainable%: 0.7032


Adding EOS to train dataset:   0%|          | 0/1373 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1373 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/152 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.375355,0.348854,0.358598,847056.000000,0.901180
2,0.321933,0.321273,0.320061,1694112.000000,0.903056
3,0.292902,0.309446,0.296578,2541168.000000,0.907589
4,0.272893,0.305086,0.293033,3388224.000000,0.907469
5,0.275493,0.304479,0.283190,4235280.000000,0.908017


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
0.275493,0.304479,5,0.283190,4235280.000000,0.908017


{'run_name': 'islamic_track_b_run2_r16_lr0.0001', 'lora_r': 16, 'lora_alpha': 32, 'lr': 0.0001, 'epochs': 5, 'eval_loss': 0.30447855591773987, 'checkpoint_dir': 'runs/islamic_track_b_run2_r16_lr0.0001/best_checkpoint'}


,run_name,lora_r,lora_alpha,lr,epochs,eval_loss,checkpoint_dir
2,islamic_track_b_run2_r16_lr0.0001,16,32,0.0001,5,0.304479,runs/islamic_track_b_run2_r16_lr0.0001/best_ch...
1,islamic_track_b_run1_r16_lr0.0002,16,32,0.0002,3,0.306591,runs/islamic_track_b_run1_r16_lr0.0002/best_ch...
0,islamic_track_b_run0_r8_lr0.0002,8,16,0.0002,3,0.318814,runs/islamic_track_b_run0_r8_lr0.0002/best_che...


## Step 8 — Checkpoint Selection and Verification

AISA layer: evaluation layer.

The best-performing checkpoint by `eval_loss` is selected, and the Step 5
comparison (tool-selection accuracy and `calculate_inheritance`
hallucination check) is re-run against it, closing the loop between
training loss and the metrics relevant to official scoring (FnAcc, ArgEM,
hallucination rate).

In [16]:
best_run = sweep_df.iloc[0]
best_checkpoint_dir = best_run["checkpoint_dir"]
print("Best run by eval_loss:", best_run["run_name"], "->", best_checkpoint_dir)

from peft import PeftModel

finetuned_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if device != "cpu" else torch.float32,
).to(device)
finetuned_model = PeftModel.from_pretrained(finetuned_base, best_checkpoint_dir).to(device)
finetuned_model.eval()

finetuned_results = []
for prompt, (gold_tool, gold_args) in zip(sample_prompts, sample_gold):
    gen = generate_call(finetuned_model, tokenizer, prompt)
    parsed = parse_function_call(gen)
    valid = validate_call(parsed)
    finetuned_results.append({
        "gold_tool": gold_tool,
        "predicted_tool": parsed["tool_name"],
        "correct_tool": parsed["tool_name"] == gold_tool,
        **valid,
    })

finetuned_df = pd.DataFrame(finetuned_results)
print(finetuned_df)
print("\nFine-tuned quick FnAcc (10-sample, informal):", finetuned_df["correct_tool"].mean())
print("Baseline quick FnAcc (10-sample, informal):    ", baseline_df["correct_tool"].mean())
print("\nHallucinated calculate_inheritance -> baseline:", baseline_df["hallucinated_inheritance"].sum(),
      "| fine-tuned:", finetuned_df["hallucinated_inheritance"].sum())


Best run by eval_loss: islamic_track_b_run2_r16_lr0.0001 -> runs/islamic_track_b_run2_r16_lr0.0001/best_checkpoint


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

             gold_tool       predicted_tool  correct_tool  valid_format  \
0  get_qibla_direction  get_qibla_direction          True          True   
1  get_qibla_direction  get_qibla_direction          True          True   
2      calculate_zakat      calculate_zakat          True          True   
3         search_quran         search_quran          True          True   
4      calculate_zakat      calculate_zakat          True          True   
5  get_qibla_direction  get_qibla_direction          True          True   
6      calculate_zakat      calculate_zakat          True          True   
7  get_qibla_direction  get_qibla_direction          True          True   
8  get_qibla_direction  get_qibla_direction          True          True   
9  get_qibla_direction  get_qibla_direction          True          True   

   valid_tool  hallucinated_inheritance  
0        True                     False  
1        True                     False  
2        True                     False  
3     

## Saving from Colab to Google Drive

In [17]:
import shutil

drive_out = Path("/content/drive/MyDrive/AISA-ArabicFC/member3_outputs")
drive_out.mkdir(parents=True, exist_ok=True)

# Copy the winning LoRA checkpoint
dest_checkpoint = drive_out / Path(best_checkpoint_dir).name
shutil.copytree(best_checkpoint_dir, dest_checkpoint, dirs_exist_ok=True)

# Copy the sweep log too, so the choice is auditable
shutil.copy("runs/sweep_results.csv", drive_out / "sweep_results.csv")

print("Saved to Drive:", dest_checkpoint)

Saved to Drive: /content/drive/MyDrive/AISA-ArabicFC/member3_outputs/best_checkpoint


## Step 9 — Handoff to Member 4

AISA layer: deployment / governance.

Deliverables:
1. Selected checkpoint directory (`best_checkpoint/`) — LoRA adapter and
   tokenizer.
2. `sweep_results.csv` — full hyperparameter sweep log.
3. This notebook, as the training record.
4. Base model identifier, final hyperparameters, and informal Step 8
   results, explicitly labeled as a small-sample preliminary check.
   Official comparison against the baseline is determined by Member 4's
   full-dev-set FnAcc/ArgEM evaluation.

In [18]:
print("=" * 60)
print("HANDOFF SUMMARY FOR MEMBER 4")
print("=" * 60)
print(f"Base model:        {BASE_MODEL}")
print(f"Best checkpoint:    {best_checkpoint_dir}")
print(f"Chosen hyperparams: r={best_run['lora_r']}, alpha={best_run['lora_alpha']}, "
      f"lr={best_run['lr']}, epochs={best_run['epochs']}")
print(f"Best eval_loss:     {best_run['eval_loss']:.4f}")
print(f"Sweep log:          runs/sweep_results.csv")
print()
print("Informal 10-sample self-check (NOT the official score):")
print(f"  Baseline FnAcc:    {baseline_df['correct_tool'].mean():.2f}")
print(f"  Fine-tuned FnAcc:  {finetuned_df['correct_tool'].mean():.2f}")
print(f"  calculate_inheritance hallucinations - baseline: {baseline_df['hallucinated_inheritance'].sum()}, "
      f"fine-tuned: {finetuned_df['hallucinated_inheritance'].sum()}")
print("=" * 60)


HANDOFF SUMMARY FOR MEMBER 4
Base model:        TuwaiqAcademy/AISA-AR-FunctionCall-Think
Best checkpoint:    runs/islamic_track_b_run2_r16_lr0.0001/best_checkpoint
Chosen hyperparams: r=16, alpha=32, lr=0.0001, epochs=5
Best eval_loss:     0.3045
Sweep log:          runs/sweep_results.csv

Informal 10-sample self-check (NOT the official score):
  Baseline FnAcc:    1.00
  Fine-tuned FnAcc:  1.00
  calculate_inheritance hallucinations - baseline: 0, fine-tuned: 0
